# 09 - Previsao operacional / hindcast

Este notebook simula o uso do **modelo final escolhido no notebook 08** em um horario historico.

Ele suporta:

- `xgboost_radar`;
- `TCN`;
- `GRU`.

A saida e propositalmente humana:

- horario em que a previsao seria emitida;
- nivel atual;
- variacao prevista em centimetros;
- nivel previsto duas horas depois;
- para redes temporais, probabilidades de subida >=1, >=2 e >=3 m;
- em um hindcast historico, comparacao posterior com o que realmente aconteceu.

Em operacao real, obviamente, a parte "o que realmente aconteceu" so fica disponivel depois.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

CWD=Path.cwd().resolve(); ROOT=CWD.parent if CWD.name.lower()=="notebooks" else CWD
PROCESSED=ROOT/"data"/"processed"; MODELS=ROOT/"models"; OUTPUTS=ROOT/"outputs"; PREDICTIONS=OUTPUTS/"predictions"
PREDICTIONS.mkdir(parents=True,exist_ok=True)
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))

meta_path=MODELS/"final_model_metadata_v6.json"
if not meta_path.exists():
    raise FileNotFoundError("Rode o notebook 08 depois de congelar a arquitetura final.")
meta=json.loads(meta_path.read_text(encoding="utf-8"))
MODEL_KIND=meta["model_kind"]; HORIZON=int(meta["horizon_min"])
print("Modelo final:",MODEL_KIND,"| horizonte:",HORIZON,"min")

## Escolha um horario historico

O exemplo abaixo usa 14/02/2024. Troque por outro timestamp da grade de 10 minutos se quiser investigar um evento especifico.

In [ ]:
PREDICTION_TIME=pd.Timestamp("2024-02-14 01:20:00")
TARGET_TIME=PREDICTION_TIME+pd.Timedelta(minutes=HORIZON)
print("Previsao emitida:",PREDICTION_TIME)
print("Horario alvo:",TARGET_TIME)

In [ ]:
master=pd.read_parquet(PROCESSED/"master_base.parquet").sort_index()
features=pd.read_parquet(PROCESSED/"features_causal.parquet").sort_index()

if PREDICTION_TIME not in master.index:
    raise KeyError("Timestamp nao encontrado na grade temporal.")

result={"prediction_time":PREDICTION_TIME,"target_time":TARGET_TIME,"model":MODEL_KIND,"horizon_min":HORIZON}

In [ ]:
if MODEL_KIND == "xgboost_radar":
    from xgboost import XGBRegressor
    df=pd.read_parquet(PROCESSED/"features_causal_radar.parquet").sort_index()
    feature_cols=[c for c in df.columns if c!="split" and not c.startswith("target_")]
    model=XGBRegressor(); model.load_model(MODELS/"final_xgboost_radar_120m.json")
    row=df.loc[[PREDICTION_TIME],feature_cols]
    pred_delta=float(model.predict(row)[0])
    stage_now=float(df.loc[PREDICTION_TIME,"stage_now"])
    result.update({"stage_now_m":stage_now,"pred_delta_m":pred_delta,"pred_delta_cm":100*pred_delta,"pred_future_stage_m":stage_now+pred_delta})

In [ ]:
if MODEL_KIND in ["TCN","GRU"]:
    import torch
    from utils.temporal_models import TemporalPreprocessorState, TemporalPreprocessor, build_temporal_frame, TCNMultiTask, GRUMultiTask

    radar=pd.read_parquet(PROCESSED/"radar_features_basic.parquet").sort_index()
    if radar.index.has_duplicates: radar=radar.groupby(level=0).mean(numeric_only=True).sort_index()
    raw,groups=build_temporal_frame(master,radar=radar)

    # Reconstrui o pre-processador final a partir do estado salvo.
    state_path=ROOT/meta["preprocessor"]
    state=TemporalPreprocessorState.from_json(state_path)
    pre=TemporalPreprocessor(groups,stage_ffill_limit=state.stage_ffill_limit); pre.state=state
    scaled=pre.transform(raw)
    pos=scaled.index.get_loc(PREDICTION_TIME); lookback=int(meta["lookback_steps"])
    if pos<lookback-1: raise RuntimeError("Nao ha historico suficiente para a janela temporal.")
    seq=scaled.iloc[pos-lookback+1:pos+1].to_numpy("float32")
    x=torch.from_numpy(seq).unsqueeze(0)
    input_dim=x.shape[-1]
    if MODEL_KIND=="TCN": model=TCNMultiTask(input_dim,channels=(64,64,96,96),kernel_size=3,dropout=.15)
    else: model=GRUMultiTask(input_dim,hidden_dim=96,num_layers=2,dropout=.15)
    ckpt=torch.load(ROOT/meta["checkpoint"],map_location="cpu",weights_only=False); model.load_state_dict(ckpt["state_dict"]); model.eval()
    with torch.no_grad(): preg,pcl=model(x)
    pred_delta=float(pre.inverse_target(preg.numpy())[0]); probs=torch.sigmoid(pcl).numpy()[0]
    probs[1]=min(probs[1],probs[0]); probs[2]=min(probs[2],probs[1])
    stage_now=float(master.loc[PREDICTION_TIME,"stage_413"])
    result.update({
        "stage_now_m":stage_now,"pred_delta_m":pred_delta,"pred_delta_cm":100*pred_delta,"pred_future_stage_m":stage_now+pred_delta,
        "p_ge_1m":float(probs[0]),"p_ge_2m":float(probs[1]),"p_ge_3m":float(probs[2]),
    })
    thresholds={float(k):float(v) for k,v in meta.get("alert_thresholds",{}).items()}
    for th,col in [(1.0,"p_ge_1m"),(2.0,"p_ge_2m"),(3.0,"p_ge_3m")]:
        if th in thresholds: result[f"alert_ge_{int(th)}m"]=bool(result[col]>=thresholds[th])

In [ ]:
# Comparacao com a realidade, apenas porque este e um hindcast historico.
if TARGET_TIME in master.index and pd.notna(master.loc[TARGET_TIME,"stage_413"]):
    real_future=float(master.loc[TARGET_TIME,"stage_413"]); real_delta=real_future-result["stage_now_m"]
    result.update({"real_delta_m":real_delta,"real_delta_cm":100*real_delta,"real_future_stage_m":real_future,"error_cm":100*(result["pred_delta_m"]-real_delta)})

summary=pd.DataFrame([result])
display(summary.T)

print("\nLEITURA HUMANA")
print(f"As {PREDICTION_TIME}, o nivel era {result['stage_now_m']:.3f} m.")
print(f"O modelo previu variacao de {result['pred_delta_cm']:+.1f} cm ate {TARGET_TIME}.")
print(f"Nivel previsto: {result['pred_future_stage_m']:.3f} m.")
if "p_ge_1m" in result:
    print(f"Risco >=1 m: {100*result['p_ge_1m']:.1f}% | >=2 m: {100*result['p_ge_2m']:.1f}% | >=3 m: {100*result['p_ge_3m']:.1f}%")
if "real_delta_cm" in result:
    print(f"Historicamente, a variacao real foi {result['real_delta_cm']:+.1f} cm; erro = {result['error_cm']:+.1f} cm.")

In [ ]:
stem=PREDICTION_TIME.strftime("%Y%m%d_%H%M")
summary.to_csv(PREDICTIONS/f"operational_hindcast_{stem}_v6.csv",index=False)
(PREDICTIONS/f"operational_hindcast_{stem}_v6.json").write_text(json.dumps({k:(str(v) if isinstance(v,pd.Timestamp) else v) for k,v in result.items()},indent=2,ensure_ascii=False),encoding="utf-8")
print("Arquivos salvos em outputs/predictions/")